<center> 

# Bioinformática Avanzada 2025

## TP Nº6: Procesamiento y Análisis Bioinformático de datos de Secuenciación Masiva
### Determinación de potenciales variantes

Daniela Rodríguez Golpe

Objetivo: 
Determinar las variantes y escribir el archivo VCF correspondiente

Una "potencial variante" es cualquier posición del genoma donde las lecturas (BAM) muestren un desacuerdo con la referencia (FASTA).

Consigna:
En base a los archivos SAM y FASTA y lo analizado en el ejercicio 61a, realice
un programa que determine las variantes y las escriba en un archivo simil VCF. Para ello
programe los siguientes módulos y luego combinelos

i) realice un programa que determine todas las potenciales variantes y para cada una guarde:
a) la profundidad de lecturas en “esa” posición del genoma de referencia.
b) la profundidad de la potencial variante observada
ii) realice un código que dados como parámetros umbrales de Profundidad de la posición y
profundidad de la variante (para variantes homo y heterocigotas) filtre las diferencias y escriba
el VCF resultante con las variantes.

El programa Módulo (I) necesita hacer lo que se conoce como un "pileup" (apilamiento) manual.

- Lógica del Programa (Módulo I)
El objetivo es iterar por cada posición de tu región de interés (chr1:209,624,476-209,628,968) y, para cada una, crear un "conteo de bases".

La lógica general es:

Iterar por cada coordenada en tu región de interés.

Para la Posición X:

Paso 1 (Referencia): Consultar tu archivo FASTA y preguntar: "¿Cuál es la base de referencia en la Posición X?". (Ej. Referencia = 'G').

Paso 2 (Lecturas): Consultar tu archivo BAM y preguntar: "De todas las lecturas que cubren la Posición X, ¿qué bases tienen?". (Ej. Lecturas = 10 'G', 4 'A').

Paso 3 (Comparar): Comparar el conteo de las lecturas (Paso 2) con la referencia (Paso 1).

Paso 4 (Decidir): Si en el conteo de lecturas existe cualquier base que no sea la de referencia (en nuestro ejemplo, hay 4 'A'), entonces la Posición X es una "potencial variante".

El resultado de tu Módulo (I) debería ser una lista de todas las posiciones que pasaron el Paso 4, junto con sus conteos.

El programa (módulo I) usaría pysam para implementar la lógica de "pileup".

Módulo II.

Al módulo I se agregan:

1- Filtros de Calidad: No queremos cualquier diferencia, solo las que tengan suficiente evidencia. Implementación de umbrales 

2- Formato VCF: Genera un archivo VCF (Variant Call Format). Este es el formato estándar en bioinformática para reportar variantes.

Lógica de los Umbrales
La mejor forma de implementar los umbrales de "profundidad para homo y heterocigotas" es usando la Fracción Alélica (Allele Fraction o AF).

Umbral de Profundidad Total (DP): El total de lecturas en esa posición (ej. DP > 10).

Umbral de Profundidad de Variante (AD): El número de lecturas que apoyan la variante (ej. AD_alt > 3).

Umbral de Fracción Alélica (AF):

Heterocigoto (0/1): La variante es ~50% de las lecturas. (Ej. AF entre 0.25 y 0.75).

Homocigoto (1/1): La variante es ~100% de las lecturas. (Ej. AF > 0.75).


In [ ]:
import pysam
import datetime

# ------ Bloque I (Configuración) --------
# --- Archivos ---
fasta_file = "/mnt/c/Backup_compu/INIDEP/CURSOS/Bioinformatica_avanzada/TP6/Homo_sapiens_assembly38.chr1.fasta"
bam_file = "/mnt/c/Backup_compu/INIDEP/CURSOS/Bioinformatica_avanzada/TP6/modulo6.sorted.bam"
vcf_output_file = "/mnt/c/Backup_compu/INIDEP/CURSOS/Bioinformatica_avanzada/TP6/variantes_filtradas.vcf"
sample_name = "mi_muestra" # Nombre que aparecerá en el VCF

# --- Región ---
region_chr = "chr1"
region_start = 209624476
region_end = 209628968

# ------ Bloque II (Parámetros de Filtrado) --------
# Umbral de Profundidad de la posición (Total Depth)
MIN_TOTAL_DEPTH = 10
# Umbral de Profundidad de la variante (Allele Depth)
MIN_ALT_ALLELE_COUNT = 3
# Umbrales de Fracción Alélica (AF) para llamar genotipo
MIN_HET_AF = 0.25 # Mínimo para ser considerado Heterocigoto
MIN_HOMO_AF = 0.75 # Mínimo para ser considerado Homocigoto

# --- Abrir los archivos ---
bam = pysam.AlignmentFile(bam_file, "rb")
fasta = pysam.FastaFile(fasta_file)

# --- Abrir el archivo VCF de salida para escribir ---
with open(vcf_output_file, "w") as vcf:
    
    # ------ Bloque III (Escribir el Header del VCF) --------
    # El header es obligatorio y describe el contenido
    today = datetime.date.today().strftime("%Y%m%d")
    vcf.write(f"##fileformat=VCFv4.2\n")
    vcf.write(f"##fileDate={today}\n")
    vcf.write(f"##source=PysamScript_ModuloII\n")
    vcf.write(f"##reference=file://{fasta_file}\n")
    vcf.write(f"##contig=<ID={region_chr},length=248956422>\n") # Longitud de chr1, ejemplo
    vcf.write(f'##INFO=<ID=DP,Number=1,Type=Integer,Description="Total Depth">\n')
    vcf.write(f'##INFO=<ID=AF,Number=A,Type=Float,Description="Allele Fraction">\n')
    vcf.write(f'##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">\n')
    vcf.write(f'##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allele Depth (ref,alt)">\n')
    vcf.write(f'##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Total Depth">\n')
    # La línea de columnas (la última del header)
    vcf.write(f"#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\t{sample_name}\n")
    
    print(f"Buscando variantes en {region_chr}:{region_start}-{region_end}")
    print(f"Filtrando con: DP > {MIN_TOTAL_DEPTH}, AD_alt > {MIN_ALT_ALLELE_COUNT}, AF_het > {MIN_HET_AF}, AF_homo > {MIN_HOMO_AF}")

    # --- Iterar por cada posición (El "Pileup") ---
    for pileupcolumn in bam.pileup(region_chr, region_start - 1, region_end):
        
        pos_actual = pileupcolumn.pos # Posición (0-based)
        total_depth = pileupcolumn.n # Profundidad total (DP)
        
        # 1. FILTRO: PROFUNDIDAD TOTAL (DP)
        # Si la profundidad es muy baja, saltamos esta posición
        if total_depth < MIN_TOTAL_DEPTH:
            continue
            
        # 2. OBTENER REFERENCIA (FASTA)
        ref_base = fasta.fetch(region_chr, pos_actual, pos_actual + 1).upper()
        
        # 3. OBTENER CONTEO DE LECTURAS (BAM)
        base_counts = {'A': 0, 'C': 0, 'G': 0, 'T': 0}
        for pileupread in pileupcolumn.pileups:
            if not pileupread.is_del and not pileupread.is_refskip:
                try:
                    base = pileupread.alignment.query_sequence[pileupread.query_position].upper()
                    if base in base_counts:
                        base_counts[base] += 1
                except (IndexError, TypeError):
                    pass # Ignorar errores raros al borde de lecturas
        
        # 4. ENCONTRAR LA VARIANTE (ALT) MÁS PROBABLE
        # Contamos la profundidad de la referencia
        ref_count = base_counts.get(ref_base, 0)
        
        # Buscamos la alternativa (ALT) con mayor conteo
        alt_base = ''
        alt_count = 0
        for base, count in base_counts.items():
            if base != ref_base and count > alt_count:
                alt_base = base
                alt_count = count
        
        # 5. FILTRO: PROFUNDIDAD DE VARIANTE (AD)
        # Si la variante no tiene suficiente soporte, la ignoramos
        if alt_count < MIN_ALT_ALLELE_COUNT:
            continue
            
        # 6. FILTRO: FRACCIÓN ALÉLICA (AF) Y GENOTIPO (GT)
        # (Aseguramos que la profundidad > 0 para evitar división por cero)
        if total_depth == 0: continue 
        
        allele_fraction = alt_count / total_depth
        
        genotype = ""
        if allele_fraction >= MIN_HOMO_AF:
            genotype = "1/1" # Homocigoto para la variante
        elif allele_fraction >= MIN_HET_AF:
            genotype = "0/1" # Heterocigoto
        else:
            # La fracción es muy baja, es ruido. Ignoramos.
            continue
            
        # 7. SI PASÓ TODOS LOS FILTROS, ESCRIBIMOS LA LÍNEA VCF
        
        chrom = region_chr
        pos = pos_actual + 1 # VCF es 1-based
        id_col = "."
        ref = ref_base
        alt = alt_base
        qual = "." # No calculamos Phred-score, así que ponemos '.'
        filter_col = "PASS" # Pasó nuestros filtros
        
        # INFO: Información general de la variante
        info = f"DP={total_depth};AF={allele_fraction:.2f}"
        
        # FORMAT: Qué datos reportamos por muestra
        format_str = "GT:AD:DP"
        
        # SAMPLE: Los datos de nuestra muestra
        sample_data = f"{genotype}:{ref_count},{alt_count}:{total_depth}"
        
        # Escribir la línea final
        vcf_line = f"{chrom}\t{pos}\t{id_col}\t{ref}\t{alt}\t{qual}\t{filter_col}\t{info}\t{format_str}\t{sample_data}\n"
        vcf.write(vcf_line)

        # Imprimimos en pantalla también para ver el progreso
        print(f"Variante FILTRADA encontrada en {chrom}:{pos} -> REF: {ref}, ALT: {alt}, GT: {genotype}, AF: {allele_fraction:.2f}, DP: {total_depth}")

# --- Cerrar archivos ---
bam.close()
fasta.close()

print("-----------------------------------------")
print(f"¡Proceso completado! VCF guardado en: {vcf_output_file}")


RESULTADO OBTENIDO:

Buscando variantes en chr1:209624476-209628968

Filtrando con: DP > 10, AD_alt > 3, AF_het > 0.25, AF_homo > 0.75

Variante FILTRADA encontrada en chr1:209625721 -> REF: G, ALT: A, GT: 0/1, AF: 0.31, DP: 159

Variante FILTRADA encontrada en chr1:209625908 -> REF: A, ALT: G, GT: 1/1, AF: 0.81, DP: 89

-----------------------------------------
¡Proceso completado! VCF guardado en: /mnt/c/Backup_compu/INIDEP/CURSOS/Bioinformatica_avanzada/TP6/variantes_filtradas.vcf